# Sesión 2.1 — Inferencia estadística

Estimación e inferencia sobre medias, varianzas y proporciones. Ejecuta las celdas en orden.

## Población, muestra y estimadores

La **población** es el conjunto sobre el que queremos concluir; la **muestra** contiene las observaciones disponibles. Un **parámetro** (por ejemplo, la media poblacional $\mu$ o la proporción $\pi$) describe la población, mientras que un **estadístico** calculado a partir de la muestra sirve para estimarlo. Como otra muestra produciría otros estadísticos, los intervalos de confianza y los contrastes cuantifican esa incertidumbre bajo sus supuestos.


## Variabilidad muestral y calidad de la muestra

Si repitiésemos el muestreo, la media y la proporción muestrales variarían de una muestra a otra. El **error estándar** estima esa variación: para la media, es aproximadamente $s/\sqrt{n}$. A igualdad de lo demás, una muestra mayor reduce el error estándar y estrecha el intervalo; pedir más confianza lo ensancha. Una mayor dispersión de los datos suele aumentar la incertidumbre.

Esto describe la **incertidumbre aleatoria**, pero no corrige una muestra sesgada. Para generalizar a una población, la muestra debe representarla razonablemente: por ejemplo, encuestar solo a clientes de una tienda o de un único plan podría dejar fuera sistemáticamente a otros perfiles. Aumentar el tamaño no elimina por sí solo ese sesgo.

## Lectura de datos

Abrimos la tabla de teléfonos móviles y la mostramos para poder inspeccionarla. Los campos están separados por `;`; `NA` y las celdas vacías representan valores ausentes.

In [ ]:
from pathlib import Path  # Permite construir rutas a los ficheros de forma portable.
import numpy as np  # Proporciona operaciones numéricas y generación de remuestras.
import pandas as pd  # Permite leer y manipular tablas de datos.
import matplotlib.pyplot as plt  # Sirve para crear gráficos.
from scipy import stats  # Aporta distribuciones y pruebas estadísticas.
from IPython.display import display  # Muestra tablas y resultados con formato en el notebook.

DATA_PATH = Path.cwd().parent.parent / "datos"  # Localiza la carpeta datos desde el directorio actual.

datos = pd.read_csv(DATA_PATH / 'Mobiles.txt', sep=';', na_values=['NA', ''])  # Lee el fichero separado por punto y coma y marca NA y celdas vacías como ausentes.
display(datos)  # Muestra la tabla completa para inspeccionar sus filas y columnas.

## Estimación de la media poblacional

Primero examinamos la distribución numérica y gráficamente. La prueba de una muestra estima la media con su intervalo de confianza (IC) y contrasta la hipótesis de que la media poblacional sea 30 (referencia del PDF) o 50.

In [ ]:
# Resumen numérico de las facturas: cuenta, media, desviación, cuartiles, mínimo y máximo.
datos["Bill"].describe()  # Calcula las estadísticas descriptivas de la columna Bill.

In [ ]:
# Histograma de las facturas: azul claro de la diapositiva y bordes gris oscuro.
fig, ax = plt.subplots(figsize=(16, 8))  # Crea una figura ancha y el eje donde dibujar.
ax.hist(datos['Bill'], bins='auto', color='#9fc3e4', edgecolor='#666666', linewidth=1.2)  # Agrupa las facturas en intervalos y muestra sus frecuencias.
ax.set_title('Distribución del importe de la factura', fontsize=20)  # Añade el título del gráfico.
ax.set_xlabel('Importe de la factura', fontsize=16)  # Etiqueta el eje horizontal con la variable medida.
ax.set_ylabel('Número de estudiantes', fontsize=16)  # Etiqueta el eje vertical con el recuento.
ax.tick_params(labelsize=14)  # Aumenta el tamaño de las etiquetas de los ejes.
plt.show()  # Renderiza el histograma.

### Intervalo de confianza para la media

Queremos estimar el importe medio de la factura en la población a partir de una muestra. El **intervalo de confianza (IC) del 95 %** proporciona un rango de valores compatibles con los datos. Si repitiésemos el procedimiento con muchas muestras, aproximadamente el 95 % de los intervalos obtenidos contendrían la verdadera media poblacional; no significa que esta media tenga un 95 % de probabilidad de estar en este intervalo concreto.

La función descarta los valores ausentes, calcula la media y utiliza la **distribución t de Student** con $n-1$ grados de libertad y el error estándar de la media. Puedes cambiar `confianza` para obtener, por ejemplo, un IC del 99 %.

In [ ]:
def intervalo_media(serie, confianza=0.95):  # Define una función para estimar la media y su intervalo bilateral.
    """Intervalo de confianza bilateral para la media de una muestra."""  # Explica el propósito de la función.
    muestra = pd.Series(serie).dropna()  # Convierte la entrada en serie y elimina valores ausentes.
    ic = stats.t.interval(confianza, df=len(muestra)-1,  # Calcula los límites con t de Student y n−1 grados de libertad.
                          loc=muestra.mean(), scale=stats.sem(muestra))  # Usa la media muestral y su error estándar.
    return pd.Series({'n': len(muestra), 'media': muestra.mean(),  # Devuelve el tamaño y la media calculados con los datos válidos.
                      'IC inferior': ic[0], 'IC superior': ic[1]})  # Añade los límites inferior y superior del intervalo.

In [ ]:
# IC del 95 % para el importe medio de la factura.
display(intervalo_media(datos['Bill']))  # Aplica la función a Bill y presenta el resultado como tabla.

#### Micropráctica: aumentar la confianza

Mira la función y el ejemplo del IC del 95 % justo arriba. Completa el hueco del código para calcular el IC del 99 %, ejecútalo y compara ambos intervalos. Antes de ejecutarlo, predice cuál será más ancho.

In [ ]:
try:
    # Calcula un IC del 99 % y compara su amplitud con la del 95 %.
    ic_95 = intervalo_media(datos['Bill'], confianza=0.95)  # Intervalo de referencia.
    confianza = ...  # Define el nuevo nivel de confianza.
    ic_nuevo = intervalo_media(datos['Bill'], confianza=...)  # Completa el argumento que falta.
    amplitud_95 = ic_95['IC superior'] - ic_95['IC inferior']  # Calcula la anchura del primer intervalo.
    amplitud_nueva = ...  # Calcula la anchura del nuevo intervalo usando sus límites.
    print(f'IC nuevo: {ic_nuevo}')  # Muestra los límites obtenidos.
    print(f'Amplitud 95 %: {amplitud_95:.2f}; amplitud nueva: {amplitud_nueva:.2f}')  # Compara ambas anchuras.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### Contraste de hipótesis para una media: prueba t de una muestra

El intervalo **estima** la media; un contraste pregunta si los datos son compatibles con un valor propuesto $\mu_0$. Planteamos una **hipótesis nula** frente a una alternativa bilateral:

$$
\left\{
\begin{aligned}
H_0 &: \mu = \mu_0 \\
H_1 &: \mu \ne \mu_0
\end{aligned}
\right.
$$

La prueba t compara la diferencia entre la media observada y $\mu_0$ con el error estándar de la media.

El **estadístico t** mide esa diferencia en unidades de error estándar. El **valor p** indica cuán extremos serían los datos observados (o más) si $H_0$ fuera cierta. Por ejemplo, con un nivel de significación de 0,05, rechazamos $H_0$ si $p<0{,}05$; si no, **no la rechazamos**, lo que no demuestra que sea verdadera. La prueba presupone observaciones independientes y, para muestras pequeñas, una distribución aproximadamente normal.

In [ ]:
def contraste_media(serie, mu=0):  # Define un contraste t bilateral frente al valor hipotético mu.
    """Contraste bilateral de la hipótesis de que la media poblacional es mu."""  # Resume la hipótesis que evalúa la función.
    muestra = pd.Series(serie).dropna()  # Conserva únicamente las observaciones no ausentes.
    resultado = stats.ttest_1samp(muestra, popmean=mu)  # Calcula el estadístico t y el valor p frente a mu.
    return pd.Series({'n': len(muestra), 'media': muestra.mean(),  # Devuelve el tamaño y la media observada.
                      'media hipotética': mu, 't': resultado.statistic,  # Incluye el valor nulo y el estadístico de prueba.
                      'p-valor': resultado.pvalue})  # Incluye el valor p bilateral para interpretar el contraste.

#### Ejemplo 1: ¿podría ser 30 la media poblacional?

Planteamos un contraste bilateral para la media de la factura:

$$
\left\{
\begin{aligned}
H_0 &: \mu = 30 \\
H_1 &: \mu \ne 30
\end{aligned}
\right.
$$

Fijamos `mu=30`, como en la referencia del PDF; los datos y las unidades son distintos. Fíjate en el valor p para decidir si los datos son compatibles con la hipótesis nula.

In [ ]:
display(contraste_media(datos['Bill'], mu=30))  # Ejecuta el contraste H₀: media poblacional = 30 y muestra sus resultados.

#### Ejemplo 2: ¿podría ser 50 la media poblacional?

Ahora contrastamos $H_0: \mu=50$ frente a $H_1: \mu\ne 50$. Solo cambia el valor propuesto mediante `mu=50`; la muestra y el procedimiento siguen siendo los mismos. Compara este valor p con el del ejemplo anterior.

In [ ]:
display(contraste_media(datos['Bill'], mu=50))  # Ejecuta el contraste H₀: media poblacional = 50 y muestra sus resultados.

#### Micropráctica: cambiar el valor de referencia

Fíjate en la función y en los contrastes anteriores. Completa el valor `mu_ref` con 40, ejecuta la prueba e interpreta el valor p con $\alpha=0{,}05$. Formula la conclusión sin afirmar que la hipótesis nula queda demostrada.

In [ ]:
try:
    # Contrasta H₀: μ = 40 frente a una alternativa bilateral.
    mu_ref = ...  # Define el valor poblacional que se contrasta.
    resultado = contraste_media(datos['Bill'], mu=...)  # Usa la función del tema y el valor anterior.
    display(resultado)  # Examina la media, t y el valor p.
    alpha = ...  # Elige el nivel de significación del 5 %.
    if resultado['p-valor'] < alpha:  # Decide usando el valor p frente a alfa.
        print('Rechazamos H₀: hay evidencia contra el valor de referencia.')
    else:
        print('No rechazamos H₀: la muestra no aporta evidencia suficiente contra él.')
    # Redacta la conclusión en contexto; no rechazar H₀ no demuestra que sea cierta.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

## Estimación de la desviación poblacional (opcional)

Usamos la fórmula del intervalo basada en cuantiles χ²: varianza, número de observaciones válidas e IC para la varianza y para la desviación típica. **Esta fórmula presupone normalidad.**

¿Se cumple la normalidad? Comparamos los cuantiles empíricos y teóricos: los datos no parecen normales si los puntos se separan de la recta.

### Alternativa: bootstrap

Generamos 10 000 muestras con reemplazo de la muestra original. Fijamos una semilla para repetir las simulaciones, representamos sus desviaciones típicas y tomamos los cuantiles 0,025 y 0,975 como IC del 95 %.

In [ ]:
s2 = datos["Bill"].var(ddof=1)  # Calcula la varianza muestral de Bill usando n−1 en el denominador.
n = datos["Bill"].notna().sum()  # Cuenta cuántas facturas tienen un valor conocido.

print(f"Varianza muestral: {s2} \nTamaño muestral: {n}")  # Muestra la varianza y el número de observaciones válidas.

In [ ]:
chi2 = stats.chi2.ppf(0.025, n-1)  # Obtiene el cuantil 2,5 % de chi-cuadrado para n−1 grados de libertad.
chi1 = stats.chi2.ppf(0.975, n-1)  # Obtiene el cuantil 97,5 % para el límite opuesto del intervalo.
LI = (n-1)*s2/chi1  # Calcula el límite inferior del intervalo para la varianza.
LS = (n-1)*s2/chi2  # Calcula el límite superior del intervalo para la varianza.
display(pd.Series({'Varianza inferior': LI, 'Varianza superior': LS,  # Prepara una tabla con los dos límites de la varianza.
                   'Desviación inferior': np.sqrt(LI), 'Desviación superior': np.sqrt(LS)}))  # Añade las raíces, que son los límites para la desviación típica.

#### Micropráctica: intervalo del 90 % para la varianza

Usa como modelo el cálculo del IC del 95 % de arriba. Completa el valor de $\alpha$ para una confianza del 90 %, ejecuta la celda y comprueba si el intervalo queda más estrecho.

In [ ]:
try:
    # Calcula el IC bilateral del 90 % para la varianza usando s2 y n de arriba.
    alpha_90 = ...  # Relaciona alfa con el nivel de confianza pedido.
    q_sup = stats.chi2.ppf(..., n-1)  # Cuantil alto de chi-cuadrado, con n−1 grados de libertad.
    q_inf = stats.chi2.ppf(..., n-1)  # Cuantil bajo de chi-cuadrado, con n−1 grados de libertad.
    li_var_90 = (n-1) * s2 / ...  # Calcula el límite inferior con el cuantil adecuado.
    ls_var_90 = (n-1) * s2 / ...  # Calcula el límite superior con el otro cuantil.
    print(f'IC del 90 % para la varianza: ({li_var_90:.2f}, {ls_var_90:.2f})')  # Presenta el intervalo para compararlo con el 95 %.
    # Comprueba que el intervalo del 90 % sea más estrecho que el del 95 %.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### Contraste de una varianza: referencia $\sigma=8$

El PDF también contrasta $H_0: \sigma^2=64$ frente a $H_1: \sigma^2\ne64$. Bajo normalidad e independencia, el estadístico $(n-1)s^2/64$ sigue una distribución $\chi^2$ con $n-1$ grados de libertad. **La cifra 8 es la referencia didáctica del PDF**; este notebook usa otro conjunto de datos, así que el resultado numérico no es directamente comparable.


In [ ]:
# Contraste bilateral H₀: sigma² = 8²; referencia ilustrativa del PDF.
sigma0 = 8  # Define la desviación típica de referencia bajo H₀.
bill_validos = datos['Bill'].dropna()  # Selecciona las facturas observadas y elimina las ausentes.
n_bill = len(bill_validos)  # Cuenta las observaciones válidas.
chi2_observado = (n_bill - 1) * bill_validos.var(ddof=1) / sigma0**2  # Calcula el estadístico chi-cuadrado bajo H₀.
gl = n_bill - 1  # Fija los grados de libertad del contraste.
p_chi2 = 2 * min(stats.chi2.cdf(chi2_observado, gl), stats.chi2.sf(chi2_observado, gl))  # Calcula el valor p bilateral sumando la cola más extrema por dos.
print(f'χ² = {chi2_observado:.4f}; gl = {gl}; p-valor bilateral = {p_chi2:.4g}')  # Muestra el estadístico, grados de libertad y valor p.
print('Se rechaza H₀ al 5 %.' if p_chi2 < 0.05 else 'No se rechaza H₀ al 5 %.')  # Decide comparando el valor p con alfa = 0,05.

In [ ]:
# Eliminamos los valores ausentes antes de comparar los cuantiles.
cuantiles, (pendiente, origen, _) = stats.probplot(datos['Bill'].dropna(), dist='norm')  # Calcula cuantiles normales teóricos y ajusta una recta a las facturas válidas.
teoricos, observados = cuantiles  # Separa las coordenadas teóricas y observadas del gráfico Q–Q.

# Mismo azul claro que el histograma anterior, con puntos más grandes y bordes negros.
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y los ejes del gráfico.
ax.scatter(teoricos, observados, s=65, color='#9fc3e4',  # Dibuja cada cuantil observado frente a su equivalente normal teórico.
           edgecolor='black', linewidth=0.8)  # Añade bordes para distinguir los puntos.
ax.plot(teoricos, pendiente * teoricos + origen, color='#800000', linewidth=1.8)  # Dibuja la recta de referencia ajustada.
ax.set_title('Gráfico de cuantiles de la factura', fontsize=20)  # Añade el título del gráfico.
ax.set_xlabel('Cuantiles teóricos de la distribución normal', fontsize=16)  # Etiqueta el eje horizontal.
ax.set_ylabel('Cuantiles observados de la factura', fontsize=16)  # Etiqueta el eje vertical.
ax.tick_params(labelsize=14)  # Aumenta el tamaño de las etiquetas de los ejes.
plt.show()  # Renderiza el gráfico Q–Q.

### Prueba de normalidad de Shapiro–Wilk

El gráfico de cuantiles permite valorar visualmente la normalidad. La **prueba de Shapiro–Wilk** la contrasta formalmente: compara la distribución de los datos con la que esperaríamos si procedieran de una población normal.

$$
\left\{
\begin{aligned}
H_0 &: \text{la distribución de las facturas es normal} \\
H_1 &: \text{la distribución de las facturas no es normal}
\end{aligned}
\right.
$$

El estadístico $W$ suele acercarse a 1 cuando los datos se ajustan a una distribución normal. Con un nivel de significación del 5 %, si el **valor p** es menor que $0{,}05$, rechazamos $H_0$; en caso contrario, no tenemos suficiente evidencia para rechazarla (**no** se demuestra que los datos sean normales). La prueba supone observaciones independientes y, como en el Q–Q plot, excluimos las facturas ausentes.

In [ ]:
# Contraste de normalidad de Shapiro–Wilk para las facturas sin valores ausentes.
w, p_valor = stats.shapiro(datos['Bill'].dropna())  # Calcula el estadístico W y el valor p de normalidad.
print(f'Shapiro–Wilk: W = {w:.4f}, valor p = {p_valor:.4g}')  # Muestra ambos resultados con redondeo legible.
if p_valor < 0.05:  # Comprueba si el valor p es menor que el nivel de significación del 5 %.
    print('Se rechaza la normalidad al nivel del 5 %.')  # Rechaza H₀ de normalidad cuando p < 0,05.
else:  # Este bloque se ejecuta cuando p es mayor o igual que 0,05.
    print('No se rechaza la normalidad al nivel del 5 %.')  # Indica que no hay evidencia suficiente para rechazar normalidad.

#### Micropráctica: probar con una muestra más pequeña

Adapta la selección aleatoria y la llamada a Shapiro–Wilk que aparecen en las celdas anteriores. Completa qué variable debes pasar a la prueba y compara su valor p con el de la muestra completa. ¿Cambia la decisión al 5 %? Un valor p alto no demuestra normalidad.

In [ ]:
try:
    # Repite la prueba de normalidad en una submuestra aleatoria reproducible.
    serie_bill = datos['Bill'].dropna()  # Excluye ausentes antes de seleccionar filas.
    n_sub = ...  # Elige el tamaño solicitado para la submuestra.
    submuestra = serie_bill.sample(n=..., random_state=12345)  # Usa el tamaño anterior y fija la semilla.
    w_sub, p_sub = stats.shapiro(...)  # Aplica Shapiro–Wilk a la submuestra, no a la tabla completa.
    print(f'n = {len(submuestra)}, W = {w_sub:.4f}, p = {p_sub:.4g}')  # Muestra los resultados.
    if p_sub < ...:  # Compara p con el nivel de significación utilizado en el ejemplo.
        print('Se rechaza la normalidad en esta submuestra.')
    else:
        print('No se rechaza la normalidad en esta submuestra.')
    # Compara con la prueba de la muestra completa y explica por qué pueden diferir.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
rng = np.random.default_rng(12345)  # Fija la semilla para que las remuestras sean reproducibles.
bill_validos = datos['Bill'].dropna().to_numpy()  # Guarda las facturas no ausentes como un arreglo de NumPy.
m = np.array([np.std(rng.choice(bill_validos, size=len(bill_validos), replace=True), ddof=1)  # Remuestrea con reemplazo y calcula la desviación típica de cada remuestra.
              for _ in range(10_000)])  # Repite el remuestreo 10 000 veces y reúne las desviaciones en un arreglo.
fig, ax = plt.subplots(figsize=(16, 8))  # Crea la figura y el eje para el histograma.
ax.hist(m, bins='auto', color='#9fc3e4', edgecolor='#666666', linewidth=1.2)  # Representa la distribución de las desviaciones bootstrap.
ax.set_title('Distribución bootstrap de la desviación típica', fontsize=20)  # Añade el título del gráfico.
ax.set_xlabel('Desviación típica bootstrap', fontsize=16)  # Etiqueta el eje horizontal.
ax.set_ylabel('Frecuencia', fontsize=16)  # Etiqueta el eje vertical con el número de remuestras.
ax.tick_params(labelsize=14)  # Aumenta el tamaño de las etiquetas de los ejes.
plt.show()  # Renderiza el histograma bootstrap.

### Intervalo de confianza bootstrap para la desviación típica

El histograma anterior muestra **10 000 desviaciones típicas muestrales**: cada una se calcula a partir de una muestra del mismo tamaño que los datos válidos, elegida **con reemplazo**. Así aproximamos cómo varía la estimación al repetir el muestreo.

Para obtener el **IC bootstrap del 95 % por el método de los percentiles**, tomamos los percentiles **2,5 % y 97,5 %** de esas 10 000 desviaciones: los valores centrales quedan entre ambos límites. El intervalo estima la **desviación típica poblacional**, no el rango que contiene el 95 % de las facturas individuales. A diferencia del IC basado en χ² de más arriba, este procedimiento no necesita suponer normalidad; sí requiere que la muestra sea representativa y sus observaciones sean independientes.

In [ ]:
limite_inferior, limite_superior = np.quantile(m, [0.025, 0.975])  # Extrae los percentiles 2,5 y 97,5 como límites del IC bootstrap del 95 %.
print(f'Percentil 2,5 % (límite inferior): {limite_inferior:.2f}')  # Muestra el extremo inferior con dos decimales.
print(f'Percentil 97,5 % (límite superior): {limite_superior:.2f}')  # Muestra el extremo superior con dos decimales.
print(f'IC bootstrap del 95 % para la desviación típica poblacional: '  # Inicia la frase que identifica el intervalo y el parámetro estimado.
      f'[{limite_inferior:.2f}, {limite_superior:.2f}]')  # Completa e imprime los dos límites del intervalo.

#### Micropráctica: intervalo bootstrap del 90 %

Consulta los percentiles usados para el IC bootstrap del 95 % y adapta el par de cuantiles para obtener el 90 % central. Completa los dos huecos, ejecuta y compara la amplitud.

In [ ]:
try:
    # Construye el IC bootstrap del 90 % y compara su amplitud con el del 95 %.
    limites_90 = np.quantile(m, [..., ...])  # Selecciona los dos cuantiles que dejan el 90 % central.
    amplitud_90 = limites_90[1] - limites_90[0]  # Calcula la anchura del intervalo del 90 %.
    amplitud_95 = ...  # Calcula la anchura del IC del 95 % obtenido en la celda anterior.
    print(f'IC bootstrap del 90 %: [{limites_90[0]:.2f}, {limites_90[1]:.2f}]')  # Muestra sus límites.
    print(f'Amplitud 90 %: {amplitud_90:.2f}; amplitud 95 %: {amplitud_95:.2f}')  # Contrasta la precisión de ambos intervalos.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

## Estimación de una proporción

Queremos estimar qué proporción de estudiantes tiene tarjeta prepago. Primero calculamos la **proporción observada**: dividimos el número de estudiantes con contrato `Prepaid` entre quienes tienen informado el tipo de contrato, **no** entre todas las filas. Después calculamos dos intervalos de confianza del 95 % y contrastamos si la proporción poblacional podría ser 0,5. Si `Plan` y `Prepaid` son las dos únicas categorías, esta hipótesis equivale a la del PDF sobre la proporción con plan fijo; los datos del notebook son distintos.

In [ ]:
e = (datos['PlanType'] == 'Prepaid').sum()  # Cuenta las filas con tarjeta prepago; las ausencias no cuentan como True.
n_plan = datos['PlanType'].notna().sum()  # Cuenta los estudiantes cuyo tipo de contrato está informado.
print(f'Prepago: {e} de {n_plan}; proporción observada = {e/n_plan:.4f}')  # Muestra el recuento y la proporción observada entre casos válidos.

### Intervalo de confianza aproximado del 95 %: método de Wilson

La proporción observada estima la proporción real de estudiantes con tarjeta prepago. El **intervalo de Wilson** da un rango de valores poblacionales compatibles con la muestra usando una aproximación basada en la distribución binomial; se comporta mejor que el intervalo aproximado más sencillo cuando la proporción está cerca de 0 o de 1. Con un nivel de confianza del 95 %, calculamos sus límites inferior y superior. El intervalo estima la proporción **poblacional**, no la proporción de una persona concreta.

In [ ]:
from statsmodels.stats.proportion import proportion_confint  # Importa funciones para intervalos de confianza de proporciones.

ic_wilson = proportion_confint(e, n_plan, alpha=0.05, method='wilson')  # Calcula el IC Wilson del 95 % con los casos y el total válidos.
print(f'IC Wilson del 95 % para la proporción de prepago: [{ic_wilson[0]:.4f}, {ic_wilson[1]:.4f}]')  # Muestra los límites del intervalo con cuatro decimales.

#### Micropráctica: reconstruir un IC de Wilson

A partir del ejemplo de `Prepaid`, construye el intervalo para la misma categoría, pero calcula de nuevo los éxitos y el total a partir de `PlanType`. Adapta el nivel de confianza al 99 % y completa los argumentos de `proportion_confint`. Luego compara su amplitud con el IC del 95 %.

In [ ]:
try:
    # Reconstruye el IC Wilson del 99 % a partir de la columna PlanType.
    planes_validos = datos['PlanType'].dropna()  # Conserva solo contratos informados.
    exitos = (planes_validos == ...).sum()  # Cuenta los contratos de la categoría que se quiere estudiar.
    total = ...  # Calcula el número de estudiantes válidos.
    alpha = ...  # Convierte la confianza solicitada en el nivel alfa.
    ic_99 = proportion_confint(exitos, total, alpha=..., method=...)  # Completa los argumentos para usar el método Wilson.
    print(f'IC del 99 %: [{ic_99[0]:.4f}, {ic_99[1]:.4f}]')  # Muestra los límites.
    # Compáralo con el IC del 95 % y comenta su amplitud.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### Intervalo de confianza binomial exacto del 95 %

El intervalo **exacto de Clopper–Pearson** utiliza la distribución binomial, sin recurrir a una aproximación normal. En muestras pequeñas es especialmente útil; suele ser más conservador (y a menudo más amplio) que el de Wilson. Seguimos estimando la misma proporción poblacional de estudiantes con tarjeta prepago, con los mismos `e` casos y `n_plan` observaciones válidas.

In [ ]:
ic_exacto = stats.binomtest(e, n_plan).proportion_ci(confidence_level=0.95, method='exact')  # Calcula el intervalo binomial exacto de Clopper–Pearson al 95 %.
print(f'IC binomial exacto del 95 % para la proporción de prepago: '  # Inicia el texto que identifica el intervalo.
      f'[{ic_exacto.low:.4f}, {ic_exacto.high:.4f}]')  # Añade e imprime los límites inferior y superior.

### Contraste binomial: ¿es 0,5 la proporción de prepago?

Los intervalos **estiman** la proporción; este contraste bilateral pregunta si los datos son compatibles con que **la mitad** de los estudiantes con contrato conocido tenga tarjeta prepago:

$$
\left\{
\begin{aligned}
H_0 &: \pi = 0{,}5 \\
H_1 &: \pi \ne 0{,}5
\end{aligned}
\right.
$$

Aquí $\pi$ es la proporción **poblacional** de prepago. La prueba binomial exacta calcula el **valor p** sin aproximación normal. Si es menor que $0{,}05$, rechazamos $H_0$; de lo contrario, **no la rechazamos** (no significa que $\pi$ sea exactamente 0,5). Se supone que las observaciones son independientes y que cada estudiante tiene la misma probabilidad de ser prepago.

In [ ]:
resultado_binomial = stats.binomtest(e, n_plan, p=0.5, alternative='two-sided')  # Contrasta exactamente si la proporción poblacional de prepago es 0,5, en ambas direcciones.
print(f'Valor p del contraste binomial bilateral (H₀: π = 0,5): {resultado_binomial.pvalue:.4g}')  # Muestra el valor p que se compara con alfa.
if resultado_binomial.pvalue < 0.05:  # Comprueba si el valor p es menor que el umbral del 5 %.
    print('Se rechaza H₀ al nivel del 5 %.')  # Rechaza la hipótesis de proporción 0,5 si el resultado es significativo.
else:  # Se ejecuta si el valor p es al menos 0,05.
    print('No se rechaza H₀ al nivel del 5 %.')  # Indica que no hay evidencia suficiente para rechazar la hipótesis nula.

#### Micropráctica: contrastar otra proporción de referencia

Revisa la llamada de `stats.binomtest` de la celda anterior. Completa `p_ref` con 0,60, ejecuta el contraste bilateral y compara su valor p con el obtenido para 0,50. Decide al nivel del 5 % y expresa la conclusión con cautela.

In [ ]:
try:
    # Repite el contraste binomial cambiando la hipótesis nula a π = 0,60.
    p_ref = ...  # Escribe el valor de referencia para la proporción poblacional.
    resultado = stats.binomtest(..., ..., p=..., alternative=...)  # Completa casos, tamaño, referencia y dirección del contraste.
    p_valor_nuevo = ...  # Extrae el valor p del resultado.
    print(f'p-valor para H₀: π = {p_ref:.2f}: {p_valor_nuevo:.4g}')  # Muestra el resultado.
    if p_valor_nuevo < ...:  # Aplica el nivel de significación del 5 %.
        print('Se rechaza H₀ al nivel del 5 %.')
    else:
        print('No se rechaza H₀ al nivel del 5 %.')
    # Compara la decisión con el contraste anterior para π = 0,50.
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

### Ejercicio 2.1

1. Lee el fichero de vuelos y guarda los datos en `vuelos_df`.
2. Estima el IC del **95 %** para la media del retraso en las llegadas (`arr_delay`) mediante una prueba t de una muestra.
3. Comprueba gráficamente que la variable no sigue una distribución normal y estima mediante bootstrap el IC del **90 %**, no del 95 %, para la desviación típica del retraso en las llegadas.
4. Estima el IC del **99 %**, no del 95 %, para la proporción de vuelos que llegan con retraso.

In [ ]:
try:
    # 2.1.1 — Lectura de los datos de vuelos desde cero.
    # Usa pd.read_csv() con DATA_PATH / 'flights.txt', separador de tabulación y los marcadores ausentes '@', 'NA' y ''.
    # Guarda el resultado en vuelos_df y comprueba la carga con display(vuelos_df.head()).
    # Revisa columnas y ausentes con vuelos_df.info() o vuelos_df.isna().sum().
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.1.2 — IC del 95 % para el retraso medio de llegada.
    # Selecciona vuelos_df['arr_delay'] y descarta ausentes con dropna().
    # Puedes reutilizar intervalo_media(..., confianza=...) o stats.ttest_1samp(...).
    # Muestra el intervalo y explica qué estima; no lo interpretes como retraso de un vuelo concreto.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.1.3 — Normalidad gráfica e IC bootstrap del 90 % para la desviación típica.
    # Usa vuelos_df['arr_delay'].dropna(); valora la normalidad con histograma y stats.probplot(..., dist='norm').
    # Reutiliza el patrón bootstrap anterior: rng.choice(..., replace=True) y np.std(..., ddof=1).
    # Ejecuta suficientes remuestras y usa np.quantile para formar el IC del 90 %.
    # Interpreta el gráfico Q–Q y comenta las limitaciones del bootstrap.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

In [ ]:
try:
    # 2.1.4 — IC del 99 % para la proporción de vuelos con retraso.
    # Define retrasado como arr_delay > 0 y usa solo observaciones conocidas.
    # Cuenta éxitos y total válido como en el ejemplo de proporciones.
    # Puedes usar proportion_confint(..., method='wilson') o stats.binomtest(...).proportion_ci(...).
    # Recuerda que alpha = 1 - confianza. Interpreta el intervalo para la proporción poblacional.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')

## Cómo cerrar una conclusión estadística

Al terminar un intervalo o un contraste, vuelve a la pregunta original y expresa el resultado en el contexto de los datos. Un intervalo resume valores plausibles para el parámetro; no describe el rango de valores individuales. En un contraste, indica la hipótesis, el nivel de significación y qué evidencia aporta el valor p.

- Si $p<\alpha$, **rechazamos $H_0$** al nivel elegido y describimos la evidencia a favor de $H_1$.
- Si $p\ge\alpha$, **no rechazamos $H_0$**: la muestra no aporta evidencia suficiente contra ella. Esto no demuestra que $H_0$ sea verdadera ni que el efecto sea exactamente cero.

La conclusión también depende del diseño, la representatividad de la muestra y los supuestos del método. La significación estadística no garantiza por sí sola que el efecto sea grande o importante en la práctica.

## Reto final integrador (opcional)

Analiza las facturas del grupo `Plan` sin copiar las celdas de resultados. Selecciona el subgrupo y, usando las funciones anteriores, calcula un IC del 99 % para la media, contrasta $H_0: \mu=50$ con un contraste bilateral y comprueba gráficamente la normalidad con un Q–Q plot. Compara el IC con el valor 50 y el valor p con $\alpha=0{,}01$: ¿coinciden las decisiones? Cierra con una conclusión en contexto y comenta qué limitaciones tienen la representatividad del subgrupo y los supuestos.

In [ ]:
try:
    # Reto final integrador (opcional): empieza aquí tu análisis desde cero.
    # Selecciona las facturas válidas de Plan con loc y dropna().
    # Usa intervalo_media(..., confianza=...) para estimar el IC del 99 %.
    # Usa contraste_media(..., mu=...) para contrastar la media frente a 50.
    # Compara el IC con el valor 50 y el p-valor con alfa = 0,01.
    # Comprueba la forma con stats.probplot(..., dist='norm') y comenta supuestos y representatividad.
    # Añade aquí tu código, con sangría dentro de este bloque.
    pass
except Exception as error:
    print('El reto no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.')
    print(f'Detalle: {type(error).__name__}: {error}')